# Feature Scaling and Gradient Descent Convergence

Feature scaling is an important preprocessing technique in machine learning, especially when using **gradient descent**. It adjusts features with different numerical ranges to comparable scales, allowing the optimization algorithm to converge faster and more efficiently.

---

## 1. Why Feature Scaling Matters

Different features can have very different numerical ranges.

For example:

```text
House size:       500 – 3000
Number of rooms:  1 – 6
```

Because of these differences, the corresponding model parameters (`w`) tend to have different magnitudes. Large-valued features generally require smaller parameters, while small-valued features may require larger parameters.

If the features are not scaled, gradient descent can become inefficient.

### Effect on the Cost Function

When features have very different scales, the contours of the cost function can become elongated ellipses:

```text
        \        /
         \      /
          \    /
           \  /
            ●  ← minimum
           /  \
          /    \
```

Gradient descent may then follow a slow **zigzag path** toward the minimum.

After feature scaling, the contours become more circular, allowing gradient descent to move more directly toward the minimum.

### Key Idea

> **Similar feature scales → better-shaped cost function → faster gradient descent convergence.**

---

# 2. Feature Scaling Methods

The goal is generally to bring features into comparable numerical ranges. A range roughly around **-1 to +1** is often desirable, although exact limits are not required.

## Scaling by Maximum Value

Each feature value is divided by the maximum value of that feature:

```text
x_scaled = x / max(x)
```

This commonly produces values between approximately `0` and `1`.

---

## Mean Normalization

Mean normalization centers the feature around zero:

```text
x_scaled = (x - mean(x)) / (max(x) - min(x))
```

This produces values typically around the range:

```text
-1  →  +1
```

The exact range depends on the data.

---

## Z-Score Normalization

Z-score normalization subtracts the mean and divides by the standard deviation:

```text
x_scaled = (x - mean(x)) / standard_deviation(x)
```

This gives the transformed feature:

- Mean ≈ `0`
- Standard deviation ≈ `1`
- Most values typically fall between approximately `-3` and `+3`



Z-score normalization is one of the most commonly used standardization techniques.

---

# 3. When Should Features Be Scaled?

Feature scaling is especially useful when features have substantially different ranges.

For example:

```text
Feature 1:  0 – 10
Feature 2:  0 – 100,000
Feature 3:  1 – 5
```

Feature 2 has a much larger scale than the other features and can contribute to a poorly conditioned optimization problem.

A practical rule is:

> **If in doubt, scale the features.**

Small differences in scale are generally not problematic, but very large differences can significantly slow gradient descent.

---

# 4. Monitoring Gradient Descent

Feature scaling improves gradient descent, but we also need a way to determine whether gradient descent is actually converging.

The main tool is the **learning curve**.

A learning curve plots:

```text
Cost J
  │\
  │ \
  │  \
  │   \
  │    \____
  │         ───
  └────────────────
        Iterations
```

### Axes

- **X-axis:** Number of gradient descent iterations
- **Y-axis:** Cost function `J`

The curve shows how the cost changes as the model parameters are repeatedly updated.

---

# 5. Recognizing Convergence

A healthy gradient descent process should generally show the cost decreasing as iterations increase.

For example:

```text
Iteration    Cost
---------    ----
0            1000
1             700
2             500
3             380
...
100            20
101            20
```

As the curve becomes flatter, additional iterations produce very little improvement.

This indicates that gradient descent is approaching convergence.

### Important Distinction

The learning curve's horizontal axis represents **iterations**, not model parameters.

The purpose of the curve is to answer:

> **Is gradient descent still making meaningful progress?**

---

# 6. Detecting Problems

The learning curve can also reveal problems with the training process.

### Cost Increasing

If the cost increases during an iteration, possible causes include:

- Learning rate `α` is too large.
- There is an implementation/coding error.

A learning rate that is too large can cause gradient descent to overshoot the minimum instead of approaching it.

### Cost Decreasing Very Slowly

If the cost decreases extremely slowly, possible causes include:

- Learning rate is too small.
- Features have very different scales.
- The optimization problem is poorly conditioned.

Feature scaling can significantly improve this situation.

---

# 7. Automatic Convergence Detection

Instead of manually inspecting the learning curve, an algorithm can use a convergence threshold.

Define a small value `ε` (epsilon).

If the improvement in cost becomes smaller than `ε`:

```text
J_previous - J_current < ε
```

the algorithm can stop training.

This avoids wasting iterations when additional optimization produces negligible improvement.

However, plotting the learning curve is still useful because it allows you to visually inspect the training behavior and identify problems.

---

# 8. Overall Workflow

A practical gradient descent workflow is:

```text
Raw Features
     │
     ▼
Check Feature Ranges
     │
     ▼
Apply Feature Scaling
     │
     ▼
Initialize Parameters
     │
     ▼
Run Gradient Descent
     │
     ▼
Monitor Cost J
     │
     ├── Cost decreases → Continue
     │
     ├── Cost increases → Check α / implementation
     │
     └── Cost change < ε → Converged
```

---

## Key Takeaways

- **Feature scaling** brings features with different numerical ranges to comparable scales.
- Large differences in feature scales can make the cost function elongated and cause gradient descent to take a slow zigzag path.
- Scaling makes the cost function better conditioned and usually allows faster convergence.
- Common techniques include:
  - Scaling by maximum value
  - Mean normalization
  - Z-score normalization
- A useful practical target is a feature range roughly around **-1 to +1**, although exact limits are not required.
- **Z-score normalization** produces features with approximately zero mean and unit standard deviation.
- A **learning curve** plots cost `J` against the number of iterations.
- A decreasing and eventually flattening learning curve indicates convergence.
- An increasing cost may indicate a learning rate that is too large or a coding error.
- An **epsilon convergence test** can automatically stop training when additional iterations provide negligible improvement.
- When unsure whether feature scaling is necessary, **scaling the features is generally a good choice**.

In [ ]:
'''
1. Feature Scaling with Mean Normalization and Z-Score
'''
import numpy as np

# Feature matrix:
# [house_size, bedrooms, age]
X = np.array([
    [2100, 4, 10],
    [1600, 3, 15],
    [2400, 4, 5],
    [1400, 2, 20]
], dtype=float)

# Mean normalization
mean = np.mean(X, axis=0)
feature_range = np.max(X, axis=0) - np.min(X, axis=0)

X_mean_normalized = (X - mean) / feature_range

print("Mean Normalization:")
print(X_mean_normalized)

# Z-score normalization
std = np.std(X, axis=0)

X_z_score = (X - mean) / std

print("\nZ-Score Normalization:")
print(X_z_score)

In [ ]:
'''
2. Gradient Descent with Learning-Curve Monitoring
'''

import numpy as np
import matplotlib.pyplot as plt


def compute_cost(X, y, w, b):
    m = len(y)

    predictions = X @ w + b
    errors = predictions - y

    return (1 / (2 * m)) * np.sum(errors ** 2)


def gradient_descent(X, y, w, b, alpha, iterations, epsilon=1e-6):
    m = len(y)
    cost_history = []

    for iteration in range(iterations):

        # Predictions
        predictions = X @ w + b

        # Gradients
        errors = predictions - y
        dw = (1 / m) * (X.T @ errors)
        db = (1 / m) * np.sum(errors)

        # Parameter update
        w -= alpha * dw
        b -= alpha * db

        # Monitor cost
        cost = compute_cost(X, y, w, b)
        cost_history.append(cost)

        # Automatic convergence check
        if iteration > 0:
            improvement = cost_history[-2] - cost

            if improvement < epsilon:
                print(f"Converged at iteration {iteration}")
                break

    return w, b, cost_history


# Example dataset
X = np.array([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0]
])

y = np.array([2.1, 4.0, 6.1, 8.0, 10.2])

# Initialize parameters
w = np.zeros(X.shape[1])
b = 0.0

# Train
w, b, cost_history = gradient_descent(
    X,
    y,
    w,
    b,
    alpha=0.01,
    iterations=1000
)

print("w =", w)
print("b =", b)

# Learning curve
plt.plot(cost_history)
plt.xlabel("Iterations")
plt.ylabel("Cost J")
plt.title("Gradient Descent Learning Curve")
plt.show()

# Gradient Descent, Feature Engineering & Polynomial Regression

## 1. Choosing the Learning Rate

The **learning rate (`α`)** controls how large each parameter update is during gradient descent. Choosing an appropriate value is essential for stable and efficient model training.

### Learning Rate Behavior

- **Too small:** The model converges slowly and requires many iterations.
- **Too large:** Updates can overshoot the minimum, causing the cost function to oscillate or increase.
- **Appropriate:** The cost function decreases consistently toward a minimum.

The goal is to find a learning rate that provides **fast convergence without destabilizing training**.

### Debugging Gradient Descent

If the cost function consistently increases, possible causes include:

- Learning rate is too large.
- The gradient update has an incorrect sign.
- There is a bug in the cost or gradient calculation.
- Parameters are being updated incorrectly.

A useful debugging technique is to temporarily use a **very small learning rate**. If the cost still does not decrease consistently, the problem is likely in the implementation rather than the learning rate.

### Practical Learning Rate Selection

A practical approach is to test several learning rates with values increasing by roughly a factor of three:

```text
α = 0.001
α = 0.003
α = 0.01
α = 0.03
α = 0.1
...
```

Select the **largest learning rate that still produces stable, consistent cost reduction**.

> A very small learning rate is useful for debugging, but it is usually inefficient for final training.

---

## 2. Feature Engineering

**Feature engineering** is the process of selecting, transforming, or combining existing features to create inputs that allow a machine learning model to better represent the underlying problem.

The quality of the features can have a major impact on model performance.

### Example: House Price Prediction

Suppose a model uses:

- Lot frontage
- Lot depth

Individually, these features may not provide the most useful representation of the property.

Instead, a new feature can be created:

```text
lot_area = frontage × depth
```

The model can then use:

```text
frontage
depth
lot_area
```

The newly engineered `lot_area` feature may provide a much stronger relationship with house price.

### Why Feature Engineering Matters

Feature engineering allows a model to work with **more meaningful representations of the original data**.

It can involve:

- Combining existing features
- Multiplying or dividing features
- Applying mathematical transformations
- Creating domain-specific measurements
- Converting raw measurements into more useful representations

The key idea is:

> **Better features can make it easier for a relatively simple model to learn a useful relationship.**

Feature engineering can also allow linear regression models to represent relationships that are **non-linear with respect to the original input**.

---

## 3. Polynomial Regression

**Polynomial regression** extends linear regression by using powers or transformations of existing features.

Instead of using only:

```text
x
```

additional features can be created:

```text
x²
x³
x⁴
...
```

For example, a quadratic model can use:

```text
x
x²
```

while a cubic model can use:

```text
x
x²
x³
```

Conceptually:

```text
Original feature
      │
      ├── x
      ├── x²
      └── x³
           │
           ▼
   Linear Regression
           │
           ▼
   Non-linear curve
```

Although the resulting relationship between `x` and the prediction can be curved, the model is still **linear in its parameters**.

### Polynomial Degree

The polynomial degree controls the flexibility of the model.

| Degree | Features | Typical Shape |
|---|---|---|
| 1 | `x` | Straight line |
| 2 | `x, x²` | Quadratic curve |
| 3 | `x, x², x³` | Cubic curve |
| Higher | `x, x², x³, ...` | Increasingly flexible |

Choosing the degree is a **model-selection decision**. A higher degree provides more flexibility but can also make the model unnecessarily complex.

---

## 4. Feature Scaling with Polynomial Features

Feature scaling becomes particularly important when polynomial features are introduced.

For example, if:

```text
x = 100
```

then:

```text
x² = 10,000
x³ = 1,000,000
```

The resulting features can have dramatically different scales.

This can make gradient descent less efficient because the optimization landscape becomes poorly scaled.

Therefore, when using polynomial features, it is often important to **scale the resulting features before training**.

The general workflow becomes:

```text
Original Data
     │
     ▼
Feature Engineering
     │
     ▼
Polynomial Features
     │
     ▼
Feature Scaling
     │
     ▼
Gradient Descent / Model Training
     │
     ▼
Prediction
```

---

## 5. Choosing Features and Transformations

Polynomial features are only one form of feature engineering.

Depending on the problem and domain knowledge, other transformations may be useful:

```text
x
x²
x³
√x
log(x)
1/x
```

The appropriate transformation depends on the relationship between the input and target.

For example, if increasing `x` produces diminishing returns, a square-root or logarithmic transformation might represent the relationship better than a high-degree polynomial.

Therefore, machine learning involves not only **training parameters**, but also deciding **which features and transformations should be provided to the model**.

---

## 6. Manual Implementation vs. Machine Learning Libraries

Understanding how algorithms work internally is important for developing a strong foundation.

For example, implementing:

- Cost functions
- Gradient descent
- Feature engineering
- Polynomial regression

manually helps build an understanding of what happens during model training.

However, real-world machine learning workflows commonly use established libraries such as **Scikit-learn** to implement these algorithms efficiently.

A practical progression is:

```text
Understand the mathematics
        ↓
Implement the algorithm manually
        ↓
Understand its behavior
        ↓
Use established ML libraries
        ↓
Apply the technique to real problems
```

---

## Key Takeaways

- The **learning rate** controls the size of gradient descent updates.
- A learning rate that is too small causes slow convergence.
- A learning rate that is too large can cause oscillation or increasing cost.
- A very small learning rate can help **debug gradient descent implementations**.
- **Feature engineering** creates more useful representations from existing data.
- Combining features can allow a model to capture more meaningful relationships.
- **Polynomial regression** uses engineered features such as `x²` and `x³` to model non-linear relationships.
- Higher-degree polynomial models provide more flexibility but can introduce unnecessary complexity.
- Polynomial features can have very different scales, making **feature scaling especially important**.
- Domain knowledge can guide the selection of useful transformations such as `√x` or `log(x)`.
- Understanding algorithms manually is valuable, while libraries such as **Scikit-learn** are commonly used for practical implementations.